In [2]:
import pandas as pd
import numpy as np

In [6]:
data = pd.read_csv("C:/Users/SaiVenkataSriramChow/Downloads/Assignment_2/Assignment_2/healthcare_data.csv")
data.head()

,Patient_ID,Name,Age,Gender,Diagnosis,Treatment,Doctor,Cost,Discount,Visit_Date
0,1,Alice,52,Female,Heart Disease,Surgery,Dr. Johnson,19019.21,NaN,2024-04-16
1,2,James,72,Female,Diabetes,Medication,Dr. Smith,12013.30,NaN,2024-04-12
2,3,Olivia,83,Male,Arthritis,Surgery,Dr. Lee,2089.50,0.0,2024-03-28
3,4,Bob,24,Female,Flu,Surgery,Dr. Johnson,13052.68,0.1,2024-11-04
4,5,Alice,2,Male,Covid-19,Therapy,Dr. Wilson,14467.78,0.0,2024-10-20


In [7]:
data.isnull().sum()

Patient_ID     0
Name           0
Age            0
Gender         0
Diagnosis      0
Treatment      0
Doctor         0
Cost           6
Discount      38
Visit_Date     0
dtype: int64

### Q1. Data Cleaning Challenge
- The **Cost** and **Discount** columns contain `"N/A"` and `NaN`.  
- Convert them to numeric format (`float`).  
- Replace missing values:  
  - **Cost** → Fill with **median cost of the patient’s Diagnosis**.  
  - **Discount** → Fill with **0**.  

In [11]:
data['Discount']= data['Discount'].fillna(0)
data['Cost'] = data['Cost'].fillna(data['Cost'].mean())
data.head()

,Patient_ID,Name,Age,Gender,Diagnosis,Treatment,Doctor,Cost,Discount,Visit_Date
0,1,Alice,52,Female,Heart Disease,Surgery,Dr. Johnson,19019.21,0.0,2024-04-16
1,2,James,72,Female,Diabetes,Medication,Dr. Smith,12013.30,0.0,2024-04-12
2,3,Olivia,83,Male,Arthritis,Surgery,Dr. Lee,2089.50,0.0,2024-03-28
3,4,Bob,24,Female,Flu,Surgery,Dr. Johnson,13052.68,0.1,2024-11-04
4,5,Alice,2,Male,Covid-19,Therapy,Dr. Wilson,14467.78,0.0,2024-10-20


In [12]:
colounmns=['Cost','Discount']
for i in colounmns:
    data[i]=data[i].astype('float')
data.dtypes

Patient_ID      int64
Name           object
Age             int64
Gender         object
Diagnosis      object
Treatment      object
Doctor         object
Cost          float64
Discount      float64
Visit_Date     object
dtype: object

### Q2. Hidden Duplicate Patients
- Notice that some patient names have extra spaces (e.g., `"John "` vs `"John"`).  
- Standardize all patient names (**strip spaces**).  
- Find if there are any **duplicate patient records** (same **Name, Age, and Diagnosis**).  
- Remove duplicates and **report how many were removed**.  



In [15]:
data['Name']=data['Name'].str.strip()
data.head()

,Patient_ID,Name,Age,Gender,Diagnosis,Treatment,Doctor,Cost,Discount,Visit_Date
0,1,Alice,52,Female,Heart Disease,Surgery,Dr. Johnson,19019.21,0.0,2024-04-16
1,2,James,72,Female,Diabetes,Medication,Dr. Smith,12013.30,0.0,2024-04-12
2,3,Olivia,83,Male,Arthritis,Surgery,Dr. Lee,2089.50,0.0,2024-03-28
3,4,Bob,24,Female,Flu,Surgery,Dr. Johnson,13052.68,0.1,2024-11-04
4,5,Alice,2,Male,Covid-19,Therapy,Dr. Wilson,14467.78,0.0,2024-10-20


In [23]:
duplicates=data.duplicated(subset=['Name','Age','Diagnosis'])
print("Total no of duplicates: ", duplicates.sum())
total_duplicates = duplicates.sum()
total_remaining= data.shape[0] - total_duplicates
print("Total no of remaining:", total_remaining)

Total no of duplicates:  2
Total no of remaining: 98


In [25]:
data = data.drop_duplicates(subset=['Name','Age','Diagnosis'], keep='first')
data.shape 

(98, 10)

### Q3. Revenue Calculation
- Create a new column **Final_Bill** using the formula:  
  `Final_Bill = Cost * (1 - Discount)`  
- Find the patient with the **maximum Final_Bill** and report:  
  - Name  
  - Diagnosis  
  - Doctor  
  - Final_Bill  

In [ ]:
data['Final_Bill'] = data['Cost'] - (1-data['Discount'])
data.head()

,Patient_ID,Name,Age,Gender,Diagnosis,Treatment,Doctor,Cost,Discount,Visit_Date
0,1,Alice,52,Female,Heart Disease,Surgery,Dr. Johnson,19019.21,0.0,2024-04-16
1,2,James,72,Female,Diabetes,Medication,Dr. Smith,12013.30,0.0,2024-04-12
2,3,Olivia,83,Male,Arthritis,Surgery,Dr. Lee,2089.50,0.0,2024-03-28
3,4,Bob,24,Female,Flu,Surgery,Dr. Johnson,13052.68,0.1,2024-11-04
4,5,Alice,2,Male,Covid-19,Therapy,Dr. Wilson,14467.78,0.0,2024-10-20


### Q4. Disease & Cost Analysis
- Group data by **Diagnosis** and compute:  
  - Total patients per disease.  
  - Average **Final_Bill** per disease.  
- Question: **Which disease costs hospitals the most on average?**

In [28]:
max_data=data.loc[data['Final_Bill'].idxmax(), ['Name', 'Diagnosis', 'Doctor', 'Final_Bill']]
print(max_data)

Name                  Alice
Diagnosis     Heart Disease
Doctor            Dr. Smith
Final_Bill         19461.91
Name: 95, dtype: object


In [39]:
Total_patients_disease=data.groupby('Diagnosis').size()
print(Total_patients_disease)
print("Total number of patients for each disease")
Average_Final_Bill_disease=data.groupby('Diagnosis').Final_Bill.mean()
print(Average_Final_Bill_disease)   

Diagnosis
Arthritis        11
Asthma           14
Cancer            7
Covid-19         12
Diabetes         12
Flu              10
Heart Disease    18
Hypertension     14
dtype: int64
Total number of patients for each disease
Diagnosis
Arthritis        10115.977505
Asthma            6862.603040
Cancer            9075.530000
Covid-19          8728.355833
Diabetes         11533.247713
Flu              10188.399511
Heart Disease     9970.673889
Hypertension     10143.768040
Name: Final_Bill, dtype: float64


### Q5. Doctor Performance
- For each **doctor**, calculate:  
  - Number of patients treated.  
  - Total revenue generated (**Final_Bill**).  
- Identify the **doctor with the highest revenue**.  

In [42]:
print("No of paitents treated by each doctor")
paitents_per_doctor=data.groupby('Doctor').size()
print(paitents_per_doctor)
print("total Revenue for each doctor")
revenue_per_doctor=round(data.groupby('Doctor').Final_Bill.sum(),2)
print(revenue_per_doctor)

No of paitents treated by each doctor
Doctor
Dr. Brown      15
Dr. Johnson    22
Dr. Lee        23
Dr. Smith      20
Dr. Wilson     18
dtype: int64
total Revenue for each doctor
Doctor
Dr. Brown      136637.31
Dr. Johnson    191468.88
Dr. Lee        246245.02
Dr. Smith      213936.09
Dr. Wilson     149101.73
Name: Final_Bill, dtype: float64


## Part C: Insights (3 Questions)

### Q6. Age vs Disease Trend
- Group patients into **age groups**:  
  - `0–18`, `19–35`, `36–60`, `61+`.  
- For each **age group**, find the **most common diagnosis**.  
- Insight: Which age group is **most vulnerable** to which disease?  

In [57]:
bins=[0,18,19,35,36,60,61]
labels=['Children','Teenage','Young_Adult','Adult','Middle_Aged','Senior_Citizen']
data['Age_group']=pd.cut(data['Age'], bins=bins, labels=labels, right=False)
most_freq_disease_for_group = data.groupby('Age_group')['Diagnosis'].agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
print(most_freq_disease_for_group)

Age_group
Children                Cancer
Teenage           Hypertension
Young_Adult           Diabetes
Adult                      NaN
Middle_Aged             Asthma
Senior_Citizen          Asthma
Name: Diagnosis, dtype: object


C:\Users\SaiVenkataSriramChow\AppData\Local\Temp\ipykernel_11712\700323056.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  most_freq_disease_for_group = data.groupby('Age_group')['Diagnosis'].agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)


### Q7. Gender Bias in Treatment
- Compare **average Final_Bill** by **Gender**.  
- Question: Which gender, on average, incurs **higher treatment costs**?  
- Suggest a **possible reason** (e.g., type of diseases, treatment choice).  

In [62]:
avg_bill_gender = data.groupby('Gender').Final_Bill.mean()
print(avg_bill_gender)
freq_disease_gender = data.groupby('Gender').Diagnosis.agg(lambda x : x.mode().iat[0] if not x.mode().empty else np.nan)
print(freq_disease_gender)


Gender
Female    9347.381972
Male      9811.416582
Name: Final_Bill, dtype: float64
Gender
Female    Heart Disease
Male       Hypertension
Name: Diagnosis, dtype: object


### Q8. Seasonal Patient Flow
- Extract **month** from `Visit_Date`.  
- Count number of patients per month.  
- Identify the **busiest month** for hospital visits.  

In [66]:
data['Month']=pd.to_datetime(data['Visit_Date']).dt.month_name()
data.head()
no_of_patients_monthly=data.groupby('Month').size()
print(no_of_patients_monthly)
print("bussiest month:", no_of_patients_monthly.idxmax())

Month
April        11
August        7
December      7
February      9
January       5
July          6
June          6
March         4
May          13
November     11
October      13
September     6
dtype: int64
bussiest month: May
